<a href="https://colab.research.google.com/github/thikhamporn0589-bit/Colab/blob/main/Lab2GE338.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [48]:
import ee
import geemap

# 🔹 ยืนยันตัวตน (ถ้ายืนยันแล้วข้าม 2 บรรทัดนี้ได้เลย)
# ee.Authenticate()
# ee.Initialize(project='ee-thikhamporn0589') # ใส่ Project ID ของคุณ

# ==========================================
# ภารกิจที่ 1: เลือกพื้นที่ศึกษา (อำเภอเมืองลำปาง)
# ==========================================
thailand = ee.FeatureCollection("FAO/GAUL/2015/level2")

lampang = thailand.filter(ee.Filter.eq('ADM1_NAME', 'Lampang'))
roi = lampang.filter(ee.Filter.eq('ADM2_NAME', 'Muang Lampang'))
roi = ee.FeatureCollection([roi.first()]) # บังคับให้เหลือแค่ 1 polygon ชัวร์ๆ

# ==========================================
# ภารกิจที่ 2: โหลด Landsat 8 และตั้งค่าช่วงเวลา/เมฆ
# ==========================================
# ฟังก์ชันปรับแก้ค่า Scale Factor ของ Landsat 8 Collection 2
def apply_scale_factors(image):
    opticalBands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    thermalBands = image.select('ST_B.*').multiply(0.00341802).add(149.0)
    return image.addBands(opticalBands, None, True).addBands(thermalBands, None, True)

# โหลดข้อมูล Landsat 8
l8_col = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
          .filterBounds(roi)
          .filterDate('2023-11-01', '2024-01-31') # ท้องฟ้าโปร่ง พ.ย. - ม.ค.
          .filter(ee.Filter.lt('CLOUD_COVER', 10)) # กรองเมฆ < 10%
          .map(apply_scale_factors)) # เรียกใช้ฟังก์ชัน Scale Factor

# สร้างภาพ Composite ด้วยค่า Median และตัดขอบเขตเฉพาะอำเภอเมือง
l8_dry = l8_col.median().clip(roi)

# ==========================================
# แสดงผลบนแผนที่
# ==========================================
Map = geemap.Map()
Map.centerObject(roi, 11)

# การแสดงผลสีจริง (True Color) ของ Landsat 8 ใช้ Band 4 (Red), 3 (Green), 2 (Blue)
vis_params = {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min': 0.0, 'max': 0.3}

Map.addLayer(l8_dry, vis_params, 'Landsat 8 True Color')
Map.addLayer(roi, {'color': 'red'}, "Mueang Lampang Boundary", False)
Map

Map(center=[18.40802438573862, 99.54591409734864], controls=(WidgetControl(options=['position', 'transparent_b…

In [50]:
import os

# 1. นำดัชนีที่คำนวณไว้มารวมเป็นภาพเดียวกัน (Image Stack)
# เพื่อให้ geemap สามารถคำนวณสถิติของทุกดัชนีได้ในการรันครั้งเดียว
indices_img = ee.Image([ndvi, ndwi])

# 2. กำหนดชื่อไฟล์และตำแหน่งที่จะเซฟตารางสถิติใน Google Colab
out_csv = '/content/mueang_lampang_zonal_stats.csv'

# 3. คำสั่งสร้าง Zonal Statistics
print("⏳ กำลังคำนวณ Zonal Statistics กรุณารอสักครู่...")
geemap.zonal_statistics(
    in_value_raster=indices_img,
    in_zone_vector=roi,          # ใช้ตัวแปร roi (ขอบเขตอำเภอเมืองลำปาง) ที่เราตัดไว้
    out_file_path=out_csv,
    stat_type='MEAN',            # หาค่าเฉลี่ย
    scale=30                     # ตั้ง Scale = 30 ตามความละเอียดของ Landsat 8
)

print(f"✅ บันทึกสถิติ Zonal Statistics เรียบร้อยแล้ว!")
print(f"📂 คุณสามารถดาวน์โหลดไฟล์ {out_csv} ได้จากแถบไฟล์ (ไอคอนโฟลเดอร์) ด้านซ้ายมือของ Colab")

⏳ กำลังคำนวณ Zonal Statistics กรุณารอสักครู่...
Computing statistics ...
Generating URL ...
Please wait ...
Data downloaded to /content/mueang_lampang_zonal_stats.csv
✅ บันทึกสถิติ Zonal Statistics เรียบร้อยแล้ว!
📂 คุณสามารถดาวน์โหลดไฟล์ /content/mueang_lampang_zonal_stats.csv ได้จากแถบไฟล์ (ไอคอนโฟลเดอร์) ด้านซ้ายมือของ Colab


In [51]:
import ee
import geemap

# 🔹 1. กำหนดพื้นที่อำเภอเมืองลำปาง (กันพัง 100%)
thailand = ee.FeatureCollection("FAO/GAUL/2015/level2")
roi = thailand.filter(ee.Filter.eq('ADM1_NAME', 'Lampang')) \
              .filter(ee.Filter.eq('ADM2_NAME', 'Muang Lampang'))
roi = ee.FeatureCollection([roi.first()])

# 🔹 2. ฟังก์ชัน Scale Factor ของ Landsat 8
def apply_scale_factors(image):
    opticalBands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    thermalBands = image.select('ST_B.*').multiply(0.00341802).add(149.0)
    return image.addBands(opticalBands, None, True).addBands(thermalBands, None, True)

# 🔹 3. สร้าง Composite 2 ช่วงเวลา
# ฝั่งซ้าย: ต้นฤดูแล้ง (พ.ย. - ม.ค.) - เมฆน้อย กรองเมฆได้
l8_early = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
            .filterBounds(roi)
            .filterDate('2023-11-01', '2024-01-31')
            .filter(ee.Filter.lt('CLOUD_COVER', 10))
            .map(apply_scale_factors)
            .median().clip(roi))

# ฝั่งขวา: พีคฤดูแล้ง (ก.พ. - เม.ย.) - ปล่อยผ่านเรื่องเมฆเพื่อป้องกัน Error จากฝุ่นควัน
l8_peak = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
           .filterBounds(roi)
           .filterDate('2024-02-01', '2024-04-30')
           .map(apply_scale_factors)
           .median().clip(roi))

# 🔹 4. แสดงผลแบบเปรียบเทียบ (Split-map)
Map = geemap.Map()
Map.centerObject(roi, 11)

# ตั้งค่า False Color (Band 5=NIR, Band 4=Red, Band 3=Green) พืชพรรณจะเป็นสีแดง
vis_false = {'bands': ['SR_B5', 'SR_B4', 'SR_B3'], 'min': 0.0, 'max': 0.3}

# สร้าง Layer สำหรับซ้ายและขวา
left_layer = geemap.ee_tile_layer(l8_early, vis_false, 'Early Dry (Nov-Jan)')
right_layer = geemap.ee_tile_layer(l8_peak, vis_false, 'Peak Dry (Feb-Apr)')

# รวมเข้าใน Split Map
Map.split_map(left_layer, right_layer)

# ตีเส้นขอบอำเภอทับลงไป
Map.addLayer(roi, {'color': 'white', 'fillColor': '00000000'}, "District Boundary")

Map

Map(center=[18.40802438573862, 99.54591409734864], controls=(ZoomControl(options=['position', 'zoom_in_text', …

In [52]:
import ee
import geemap

# 🔹 1. กำหนดพื้นที่อำเภอเมืองลำปาง (กันพัง 100%)
thailand = ee.FeatureCollection("FAO/GAUL/2015/level2")
roi = thailand.filter(ee.Filter.eq('ADM1_NAME', 'Lampang')) \
              .filter(ee.Filter.eq('ADM2_NAME', 'Muang Lampang'))
roi = ee.FeatureCollection([roi.first()])

# 🔹 2. ฟังก์ชัน Scale Factor ของ Landsat 8
def apply_scale_factors(image):
    opticalBands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    thermalBands = image.select('ST_B.*').multiply(0.00341802).add(149.0)
    return image.addBands(opticalBands, None, True).addBands(thermalBands, None, True)

# 🔹 3. โหลดภาพ Landsat 8 ช่วงต้นฤดูแล้ง
l8_dry = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
          .filterBounds(roi)
          .filterDate('2023-11-01', '2024-01-31')
          .filter(ee.Filter.lt('CLOUD_COVER', 10))
          .map(apply_scale_factors)
          .median().clip(roi))

# 🔹 4. คำนวณ Index 2 ตัว (NDVI และ NDWI)
# Landsat 8: NDVI = (B5 - B4) / (B5 + B4), NDWI = (B3 - B5) / (B3 + B5)
ndvi = l8_dry.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
ndwi = l8_dry.normalizedDifference(['SR_B3', 'SR_B5']).rename('NDWI')

# 🔹 5. แสดงผลเปรียบเทียบความสัมพันธ์ด้วย Split-map
Map = geemap.Map()
Map.centerObject(roi, 11)

# ตั้งค่าสีสำหรับการแสดงผล (Visualization Parameters)
vis_ndvi = {'min': 0, 'max': 0.6, 'palette': ['red', 'yellow', 'green']}
vis_ndwi = {'min': -0.3, 'max': 0.3, 'palette': ['red', 'white', 'blue']}

# สร้าง Layer ซ้าย (NDVI) และ ขวา (NDWI)
left_layer = geemap.ee_tile_layer(ndvi, vis_ndvi, 'NDVI (Vegetation)')
right_layer = geemap.ee_tile_layer(ndwi, vis_ndwi, 'NDWI (Water/Moisture)')

# เพิ่มลงใน Split Map
Map.split_map(left_layer, right_layer)

# ตีเส้นขอบอำเภอเมืองลำปางทับลงไปให้เห็นอาณาเขตชัดเจน
Map.addLayer(roi, {'color': 'black', 'fillColor': '00000000'}, "Mueang Lampang Boundary")

Map

Map(center=[18.40802438573862, 99.54591409734864], controls=(ZoomControl(options=['position', 'zoom_in_text', …

In [53]:
import ee
import geemap

# 🔹 1. กำหนดพื้นที่ "อำเภอเมืองลำปาง"
thailand = ee.FeatureCollection("FAO/GAUL/2015/level2")
roi = thailand.filter(ee.Filter.eq('ADM1_NAME', 'Lampang')) \
              .filter(ee.Filter.eq('ADM2_NAME', 'Muang Lampang'))
roi = ee.FeatureCollection([roi.first()])

# 🔹 2. ฟังก์ชันปรับค่า Scale Factor ของ Landsat 8
def apply_scale_factors(image):
    opticalBands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    thermalBands = image.select('ST_B.*').multiply(0.00341802).add(149.0)
    return image.addBands(opticalBands, None, True).addBands(thermalBands, None, True)

# 🔹 3. สร้าง Composite 2 ช่วงเวลา
# ช่วงที่ 1: ต้นฤดูแล้ง (พ.ย. 23 - ม.ค. 24)
l8_early = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
            .filterBounds(roi)
            .filterDate('2023-11-01', '2024-01-31')
            .filter(ee.Filter.lt('CLOUD_COVER', 10))
            .map(apply_scale_factors).median().clip(roi))

# ช่วงที่ 2: พีคฤดูแล้ง (ก.พ. 24 - เม.ย. 24)
l8_peak = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
           .filterBounds(roi)
           .filterDate('2024-02-01', '2024-04-30')
           .map(apply_scale_factors).median().clip(roi))

# 🔹 4. คำนวณ Index ของทั้ง 2 ช่วงเวลา
# NDVI (พืชพรรณ)
ndvi_early = l8_early.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI_Early')
ndvi_peak = l8_peak.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI_Peak')

# NDWI (แหล่งน้ำ/ความชื้น)
ndwi_early = l8_early.normalizedDifference(['SR_B3', 'SR_B5']).rename('NDWI_Early')
ndwi_peak = l8_peak.normalizedDifference(['SR_B3', 'SR_B5']).rename('NDWI_Peak')

# 🔹 5. วิเคราะห์การเปลี่ยนแปลง (Change Detection: Peak - Early)
ndvi_change = ndvi_peak.subtract(ndvi_early).rename('NDVI_Change')
ndwi_change = ndwi_peak.subtract(ndwi_early).rename('NDWI_Change')

# 🔹 6. การแสดงผล (Visualization)
Map = geemap.Map()
Map.centerObject(roi, 11)

# ตั้งค่าสีสำหรับแผนที่ Change (สีแดง = ลดลง/แย่ลง, สีขาว = คงที่, สีเขียว/น้ำเงิน = เพิ่มขึ้น)
vis_change_ndvi = {'min': -0.3, 'max': 0.3, 'palette': ['red', 'white', 'green']}
vis_change_ndwi = {'min': -0.3, 'max': 0.3, 'palette': ['red', 'white', 'blue']}

# เพิ่มเลเยอร์ปกติ (ซ่อนไว้ก่อน)
Map.addLayer(ndvi_early, {'min': 0, 'max': 0.6, 'palette': ['red', 'yellow', 'green']}, 'NDVI Early Dry', False)
Map.addLayer(ndvi_peak, {'min': 0, 'max': 0.6, 'palette': ['red', 'yellow', 'green']}, 'NDVI Peak Dry', False)

# 🔥 เพิ่มเลเยอร์ Change Detection (เปิดโชว์เป็นหลัก)
Map.addLayer(ndvi_change, vis_change_ndvi, '🔥 NDVI Change (Loss/Gain)')
Map.addLayer(ndwi_change, vis_change_ndwi, '💧 NDWI Change (Loss/Gain)')

# เพิ่มเส้นขอบอำเภอ
Map.addLayer(roi, {'color': 'black', 'fillColor': '00000000'}, "Mueang Lampang Boundary")

Map

Map(center=[18.40802438573862, 99.54591409734864], controls=(WidgetControl(options=['position', 'transparent_b…